In [ ]:
!pip install qdrant-client groq sentence-transformers dspy-ai==2.4.12 fastembed gradio --upgrade


# Upload dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv("healthcare_dataset.csv")

In [ ]:
def format_row(row):
    return (
        f"Name: {row['Name']}, Age: {row['Age']}, Gender: {row['Gender']}, "
        f"Blood Type: {row['Blood Type']}, Medical Condition: {row['Medical Condition']}, "
        f"Date of Admission: {row['Date of Admission']}, Doctor: {row['Doctor']}, "
        f"Hospital: {row['Hospital']}, Insurance Provider: {row['Insurance Provider']}, "
        f"Billing Amount: {row['Billing Amount']}, Room Number: {row['Room Number']}, "
        f"Admission Type: {row['Admission Type']}, Discharge Date: {row['Discharge Date']}, "
        f"Medication: {row['Medication']}, Test Results: {row['Test Results']}"
        "\n\n".lower()
    )

df['formatted_text'] = df.apply(format_row, axis=1)

text_data = df['formatted_text'].tolist()

In [ ]:
from random import shuffle
sampled_dataset = text_data[:128]
shuffle(sampled_dataset)

In [ ]:
sampled_dataset[:5]

# generate embeddings in order to store them in a vector DB

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device='cuda')
vectors = model.encode(sampled_dataset)

In [ ]:
vectors[0].shape

# Qdrant cloud

In [ ]:
import os
os.environ['QDRANT__SERVICE__API_KEY']= '<YOUR QDRANT SERVICE API>'
os.environ['QDRANT__SERVICE__JWT_RBAC']='true'

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams


# Initialize the client

client = QdrantClient(
    url= '<YOUR QDRANT CLIENT URL>',
    # url='http://localhost:6333',
    api_key=os.environ['QDRANT__SERVICE__API_KEY'],
)

In [ ]:
client.recreate_collection(
    collection_name="phi_data",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

client.upload_collection(
    collection_name="phi_data",
    ids=[i for i in range(len(sampled_dataset))],
    vectors=vectors,
    parallel=4,
    max_retries=3,
)

In [ ]:
def get_context(text):
    query_vector = model.encode(text)

    hits = client.search(
        collection_name="phi_data",
        query_vector=query_vector,
        limit=3
    )
    s=''
    for x in [sampled_dataset[i.id] for i in hits]:
        s = s + x
    return s

# Dspy pipeline

In [ ]:
import dspy
from dspy.retrieve.qdrant_rm import QdrantRM

In [ ]:
qdrant_retriever_model = QdrantRM("phi_data", client, k=3)

In [ ]:
import dspy

llama3 = dspy.GROQ(model='llama3-8b-8192', api_key ="<YOUR GROQ API KEY>" )

In [ ]:
dspy.settings.configure(rm=qdrant_retriever_model, lm=llama3)

class GenerateAnswer(dspy.Signature):
    """Answer questions with logical factoid answers."""

    context = dspy.InputField(desc="will contain PHI medical data of patients matched with the query")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="an answer between 10 to 20 words")

In [ ]:
class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()

        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        context = get_context(question)
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

# Test Inference

In [ ]:
uncompiled_rag = RAG()

In [ ]:
def respond(query):
    response = uncompiled_rag(query)
    return response.answer

In [ ]:
respond("steven james")

In [ ]:
llama3.inspect_history(n=1)

# Gradio UI

In [ ]:
import gradio as gr


with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    def respond(query, chat_history):
        response = uncompiled_rag(query)
        chat_history.append((query, response.answer))
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

In [ ]:
demo.launch()
# demo.launch(share=True) if using colab

In [ ]:
sampled_dataset[67]